# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

A feature vector is the collection of input variables that will later be used by a machine learning model. For the Refresh / Content Opportunity Scoring lane, the features should describe the current performance of a content page without using any future information.

I selected features from Google Search Console and Google Analytics because they represent search visibility, user engagement, and website traffic. Missing values are filled with zero because missing metrics generally indicate no recorded activity rather than unknown information.

No future-window or label-derived information is included in the feature vector.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

# Login
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# Load dataset
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

import pandas as pd

# Only first 50,000 rows
df = pd.DataFrame(dataset["train"][:50000])

df.head()

# Feature columns
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "scroll_events"
]

# Build feature vector
feature_df = df[features].copy()

# Fill missing values
feature_df = feature_df.fillna(0)

print("Feature Vector Shape:", feature_df.shape)
feature_df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Feature Vector Shape: (50000, 9)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,scroll_events
0,30,0,3.833333,0,0,0,0,0,0
1,5,0,71.600000,0,0,0,0,0,0
2,1,0,34.000000,0,0,0,0,0,0
3,6,0,23.333333,0,0,0,0,0,0
4,5,0,17.800000,0,0,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature vector consists of numerical search performance and engagement metrics collected from Google Search Console (GSC) and Google Analytics 4 (GA4). These features describe the current state of a content page before any prediction is made.

Missing values are filled with **0**, since missing metrics generally indicate that no activity was recorded for that metric. All selected features are available **before the prediction time**, making them safe to use for model training.

| Feature | Meaning | Missing Value Handling | Available Before Prediction? |
|----------|---------|------------------------|------------------------------|
| **gsc_impressions** | Number of times the page appeared in Google Search results | Fill with 0 | Yes |
| **gsc_clicks** | Number of clicks received from Google Search | Fill with 0 | Yes |
| **gsc_avg_position** | Average ranking position in Google Search | Fill with 0 | Yes |
| **ga4_pageviews** | Total number of page views | Fill with 0 | Yes |
| **ga4_sessions** | Total website sessions | Fill with 0 | Yes |
| **ga4_users** | Number of unique users | Fill with 0 | Yes |
| **ga4_engaged_sessions** | Number of engaged sessions | Fill with 0 | Yes |
| **ga4_total_engagement_sec** | Total user engagement time (seconds) | Fill with 0 | Yes |
| **scroll_events** | Number of recorded scroll events | Fill with 0 | Yes |

### Summary

- All selected features are **numerical**.
- No categorical features are used in this notebook.
- Missing values are replaced with **0**.
- Every feature represents information available before making a prediction, so none of them introduce data leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Display feature information

print("Feature Vector Shape:", feature_df.shape)

print("\nSelected Features:")
print(feature_df.columns.tolist())

print("\nMissing Values:")
print(feature_df.isnull().sum())

print("\nData Types:")
print(feature_df.dtypes)

Feature Vector Shape: (50000, 9)

Selected Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'scroll_events']

Missing Values:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
scroll_events               0
dtype: int64

Data Types:
gsc_impressions               int64
gsc_clicks                    int64
gsc_avg_position            float64
ga4_pageviews                 int64
ga4_sessions                  int64
ga4_users                     int64
ga4_engaged_sessions          int64
ga4_total_engagement_sec      int64
scroll_events                 int64
dtype: object


## 3. The leakage hunt

Data leakage occurs when information that would not be available at prediction time is accidentally used during training.

For this project, I checked that none of the selected features are derived from future outcomes or directly contain the target information.

## Result

The selected feature vector contains only historical search and engagement metrics that are available before the prediction time.

I verified that no future outcome variables, target labels, or label-derived columns were included in the feature vector. Therefore, the selected features do not introduce data leakage and are suitable for building an ML model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether any suspicious columns exist

possible_leakage = [
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "refresh_label"
]

for column in possible_leakage:
    if column in df.columns:
        print(column, "FOUND (Potential Leakage)")
    else:
        print(column, "Not Present")
print("Selected Features")
print(feature_df.columns.tolist())

trend_direction Not Present
trend_pct Not Present
future_clicks Not Present
future_impressions Not Present
refresh_label Not Present
Selected Features
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'scroll_events']


## 4. What I excluded and why


The following fields are deliberately excluded from the feature vector because they either identify records, represent metadata instead of content performance, or could introduce data leakage.

| **Field** | **Reason for Exclusion** |
|-----------|--------------------------|
| `client_hash_id` | Identifier only; does not describe page performance. |
| `content_hash_id` | Identifier only; not a predictive feature. |
| `report_date` | Used for filtering the data by time; not used as a predictive feature. |
| `client_has_gsc` | Metadata indicating Search Console availability rather than page performance. |
| `client_has_ga4` | Metadata indicating Google Analytics availability rather than page performance. |
| `gsc_data_available` | Availability flag; does not measure user or search behavior. |
| `ga4_data_available` | Availability flag; does not measure user or search behavior. |
| Any future outcome columns | Excluded to prevent data leakage from future information. |
| Label-derived columns | Excluded because they would allow the model to directly infer the target, resulting in unrealistically high performance. |

### Summary

The excluded fields fall into three categories:

- **Identifiers** (`client_hash_id`, `content_hash_id`) that uniquely identify records but provide no predictive information.
- **Metadata and availability flags** (`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`) that describe data availability rather than content performance.
- **Leakage-prone fields**, including future outcome variables and label-derived columns, which could expose information unavailable at prediction time and lead to misleading model performance.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

print("Excluded Fields")
for field in excluded:
    print("-", field)

Excluded Fields
- client_hash_id
- content_hash_id
- report_date
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.